# Online Retail — Dataset Understanding

**Project:** E-commerce Data Analysis Pipeline  
**Stage:** Dataset Understanding & Data Quality Assessment

## Mục tiêu

Notebook này chỉ dùng để hiểu **raw dataset trước khi cleaning**.

Trong bước này:
- kiểm tra schema và datatype;
- khảo sát missing values và duplicates;
- khảo sát `Quantity`, `UnitPrice`, cancellations;
- hiểu invoice, product, customer, country và date range;
- tạo **Data Quality Summary**.

> Không xóa hoặc sửa dữ liệu trong notebook này.


In [1]:
import pandas as pd

from ecommerce_analysis.config import RAW_DATA_PATH
from ecommerce_analysis.data_loader import load_raw_data

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

df = load_raw_data(RAW_DATA_PATH)

print(f"Raw path: {RAW_DATA_PATH}")
print(f"Shape: {df.shape}")


Raw path: /home/namdp/Documents/Projects/ecommerce-data-analysis-pipeline/data/raw/online_retail.csv
Shape: (541909, 8)


## 1. Dataset preview

In [2]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [3]:
df.sample(10, random_state=42)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
209268,555200,71459,HANGING JAM JAR T-LIGHT HOLDER,24,6/1/2011 12:05,0.85,17315.0,United Kingdom
207108,554974,21128,GOLD FISHING GNOME,4,5/27/2011 17:14,6.95,14031.0,United Kingdom
167085,550972,21086,SET/6 RED SPOTTY PAPER CUPS,4,4/21/2011 17:05,0.65,14031.0,United Kingdom
471836,576652,22812,PACK 3 BOXES CHRISTMAS PANETTONE,3,11/16/2011 10:39,1.95,17198.0,United Kingdom
115865,546157,22180,RETROSPOT LAMP,2,3/10/2011 8:40,9.95,13502.0,United Kingdom
465024,576200,82482,WOODEN PICTURE FRAME WHITE FINISH,2,11/14/2011 12:14,2.95,15572.0,United Kingdom
477777,577076,22614,PACK OF 12 SPACEBOY TISSUES,12,11/17/2011 15:08,0.39,14362.0,United Kingdom
367855,568909,22596,CHRISTMAS STAR WISH LIST CHALKBOARD,12,9/29/2011 13:38,1.25,16818.0,United Kingdom
491657,578072,21109,LARGE CAKE TOWEL CHOCOLATE SPOTS,1,11/22/2011 16:02,6.75,17759.0,United Kingdom
269641,560491,23297,SET 40 HEART SHAPE PETIT FOUR CASES,2,7/19/2011 10:51,1.65,12415.0,Australia


## 2. Schema & data types

In [4]:
df.columns.tolist()

['InvoiceNo',
 'StockCode',
 'Description',
 'Quantity',
 'InvoiceDate',
 'UnitPrice',
 'CustomerID',
 'Country']

In [5]:
df.dtypes

InvoiceNo          str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
UnitPrice      float64
CustomerID     float64
Country            str
dtype: object

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 33.1 MB


## 3. Missing values

In [7]:
missing = df.isna().sum().to_frame("missing_count")
missing["missing_percent"] = (missing["missing_count"] / len(df) * 100).round(2)
missing.sort_values("missing_count", ascending=False)


,missing_count,missing_percent
CustomerID,135080,24.93
Description,1454,0.27
StockCode,0,0.00
InvoiceNo,0,0.00
Quantity,0,0.00
InvoiceDate,0,0.00
UnitPrice,0,0.00
Country,0,0.00


In [8]:
df[df["CustomerID"].isna()].head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,12/1/2010 11:52,0.00,NaN,United Kingdom
1443,536544,21773,DECORATIVE ROSE BATHROOM BOTTLE,1,12/1/2010 14:32,2.51,NaN,United Kingdom
1444,536544,21774,DECORATIVE CATS BATHROOM BOTTLE,2,12/1/2010 14:32,2.51,NaN,United Kingdom
1445,536544,21786,POLKADOT RAIN HAT,4,12/1/2010 14:32,0.85,NaN,United Kingdom
1446,536544,21787,RAIN PONCHO RETROSPOT,2,12/1/2010 14:32,1.66,NaN,United Kingdom
1447,536544,21790,VINTAGE SNAP CARDS,9,12/1/2010 14:32,1.66,NaN,United Kingdom
1448,536544,21791,VINTAGE HEADS AND TAILS CARD GAME,2,12/1/2010 14:32,2.51,NaN,United Kingdom
1449,536544,21801,CHRISTMAS TREE DECORATION WITH BELL,10,12/1/2010 14:32,0.43,NaN,United Kingdom
1450,536544,21802,CHRISTMAS TREE HEART DECORATION,9,12/1/2010 14:32,0.43,NaN,United Kingdom
1451,536544,21803,CHRISTMAS TREE STAR DECORATION,11,12/1/2010 14:32,0.43,NaN,United Kingdom


In [9]:
df[df["Description"].isna()].head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,12/1/2010 11:52,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,12/1/2010 14:32,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,12/1/2010 14:33,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,12/1/2010 14:33,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,12/1/2010 14:34,0.0,NaN,United Kingdom
1988,536550,85044,NaN,1,12/1/2010 14:34,0.0,NaN,United Kingdom
2024,536552,20950,NaN,1,12/1/2010 14:34,0.0,NaN,United Kingdom
2025,536553,37461,NaN,3,12/1/2010 14:35,0.0,NaN,United Kingdom
2026,536554,84670,NaN,23,12/1/2010 14:35,0.0,NaN,United Kingdom
2406,536589,21777,NaN,-10,12/1/2010 16:50,0.0,NaN,United Kingdom


## 4. Duplicate rows

In [10]:
duplicate_count = int(df.duplicated().sum())
duplicate_count


5268

In [11]:
df[df.duplicated(keep=False)].head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,12/1/2010 11:45,4.95,17908.0,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,12/1/2010 11:45,2.10,17908.0,United Kingdom
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,12/1/2010 11:45,1.25,17908.0,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,12/1/2010 11:45,1.25,17908.0,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,12/1/2010 11:45,2.95,17908.0,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,12/1/2010 11:45,2.10,17908.0,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,12/1/2010 11:45,2.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,12/1/2010 11:45,4.95,17908.0,United Kingdom
548,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,12/1/2010 11:49,2.95,17920.0,United Kingdom
555,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,12/1/2010 11:49,2.95,17920.0,United Kingdom


## 5. Quantity analysis

In [12]:
df["Quantity"].describe()

count    541909.000000
mean          9.552250
std         218.081158
min      -80995.000000
25%           1.000000
50%           3.000000
75%          10.000000
max       80995.000000
Name: Quantity, dtype: float64

In [13]:
pd.Series({
    "quantity_negative": int((df["Quantity"] < 0).sum()),
    "quantity_zero": int((df["Quantity"] == 0).sum()),
    "quantity_positive": int((df["Quantity"] > 0).sum()),
})


quantity_negative     10624
quantity_zero             0
quantity_positive    531285
dtype: int64

In [14]:
df.loc[
    df["Quantity"] < 0,
    ["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "CustomerID", "Country"],
].head(30)


,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,0.29,17548.0,United Kingdom
238,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,0.29,17548.0,United Kingdom
239,C536391,21484,CHICK GREY HOT WATER BOTTLE,-12,3.45,17548.0,United Kingdom
240,C536391,22557,PLASTERS IN TIN VINTAGE PAISLEY,-12,1.65,17548.0,United Kingdom
241,C536391,22553,PLASTERS IN TIN SKULLS,-24,1.65,17548.0,United Kingdom
939,C536506,22960,JAM MAKING SET WITH JARS,-6,4.25,17897.0,United Kingdom


## 6. Cancelled invoices

In [15]:
cancel_mask = df["InvoiceNo"].astype(str).str.startswith("C")

cancelled_row_count = int(cancel_mask.sum())
cancelled_invoice_count = int(df.loc[cancel_mask, "InvoiceNo"].nunique())

print("Cancelled rows:", cancelled_row_count)
print("Cancelled invoices:", cancelled_invoice_count)


Cancelled rows: 9288
Cancelled invoices: 3836


In [16]:
df.loc[cancel_mask].head(20)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,12/1/2010 9:41,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,12/1/2010 9:49,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,12/1/2010 10:24,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,12/1/2010 10:24,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,12/1/2010 10:24,0.29,17548.0,United Kingdom
238,C536391,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,12/1/2010 10:24,0.29,17548.0,United Kingdom
239,C536391,21484,CHICK GREY HOT WATER BOTTLE,-12,12/1/2010 10:24,3.45,17548.0,United Kingdom
240,C536391,22557,PLASTERS IN TIN VINTAGE PAISLEY,-12,12/1/2010 10:24,1.65,17548.0,United Kingdom
241,C536391,22553,PLASTERS IN TIN SKULLS,-24,12/1/2010 10:24,1.65,17548.0,United Kingdom
939,C536506,22960,JAM MAKING SET WITH JARS,-6,12/1/2010 12:38,4.25,17897.0,United Kingdom


In [17]:
negative_quantity_mask = df["Quantity"] < 0

pd.crosstab(
    cancel_mask.rename("is_cancelled"),
    negative_quantity_mask.rename("quantity_negative"),
)


quantity_negative,False,True
is_cancelled,,
False,531285,1336
True,0,9288


### Tự trả lời

1. Mọi cancelled invoice có `Quantity < 0` không?
2. Mọi `Quantity < 0` có thuộc cancelled invoice không?
3. Nếu có ngoại lệ, chúng là gì?


## 7. UnitPrice analysis

In [18]:
df["UnitPrice"].describe()

count    541909.000000
mean          4.611114
std          96.759853
min      -11062.060000
25%           1.250000
50%           2.080000
75%           4.130000
max       38970.000000
Name: UnitPrice, dtype: float64

In [19]:
pd.Series({
    "unit_price_negative": int((df["UnitPrice"] < 0).sum()),
    "unit_price_zero": int((df["UnitPrice"] == 0).sum()),
    "unit_price_positive": int((df["UnitPrice"] > 0).sum()),
})


unit_price_negative         2
unit_price_zero          2515
unit_price_positive    539392
dtype: int64

In [20]:
df.loc[
    df["UnitPrice"] <= 0,
    ["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "CustomerID", "Country"],
].head(30)


,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,0.0,NaN,United Kingdom
1988,536550,85044,NaN,1,0.0,NaN,United Kingdom
2024,536552,20950,NaN,1,0.0,NaN,United Kingdom
2025,536553,37461,NaN,3,0.0,NaN,United Kingdom
2026,536554,84670,NaN,23,0.0,NaN,United Kingdom
2406,536589,21777,NaN,-10,0.0,NaN,United Kingdom


## 8. Invoice analysis

In [21]:
unique_invoice_count = int(df["InvoiceNo"].nunique())
unique_invoice_count


25900

In [22]:
invoice_sizes = (
    df.groupby("InvoiceNo")
    .size()
    .sort_values(ascending=False)
)
invoice_sizes.head(20)


InvoiceNo
573585    1114
581219     749
581492     731
580729     721
558475     705
579777     687
581217     676
537434     675
580730     662
538071     652
580367     650
580115     645
581439     635
580983     629
578344     622
538349     620
578347     606
537638     601
537237     597
536876     593
dtype: int64

> **1 row không đồng nghĩa với 1 order.** Một invoice có thể chứa nhiều product lines.

## 9. Product analysis

In [23]:
unique_stock_codes = int(df["StockCode"].nunique())
unique_descriptions = int(df["Description"].nunique(dropna=True))

print("Unique StockCode:", unique_stock_codes)
print("Unique Description:", unique_descriptions)


Unique StockCode: 4070
Unique Description: 4223


In [24]:
description_per_stock = (
    df.groupby("StockCode")["Description"]
    .nunique(dropna=True)
    .sort_values(ascending=False)
)
description_per_stock.head(20)


StockCode
20713     8
23084     7
85175     6
21830     6
21181     5
85172     5
72807A    5
23343     5
23131     5
46000S    4
22121     4
23196     4
21829     4
23203     4
21823     4
35965     4
37327     4
22719     4
22734     4
22812     4
Name: Description, dtype: int64

## 10. Customer analysis

In [25]:
unique_customer_count = int(df["CustomerID"].nunique(dropna=True))

print("Unique customers:", unique_customer_count)
print("Rows with CustomerID:", int(df["CustomerID"].notna().sum()))
print("Rows missing CustomerID:", int(df["CustomerID"].isna().sum()))


Unique customers: 4372
Rows with CustomerID: 406829
Rows missing CustomerID: 135080


## 11. Country analysis

In [26]:
country_count = int(df["Country"].nunique(dropna=True))
country_count


38

In [27]:
df["Country"].value_counts().head(20)

Country
United Kingdom     495478
Germany              9495
France               8557
EIRE                 8196
Spain                2533
Netherlands          2371
Belgium              2069
Switzerland          2002
Portugal             1519
Australia            1259
Norway               1086
Italy                 803
Channel Islands       758
Finland               695
Cyprus                622
Sweden                462
Unspecified           446
Austria               401
Denmark               389
Japan                 358
Name: count, dtype: int64

## 12. Date range

In [28]:
invoice_dates = pd.to_datetime(
    df["InvoiceDate"],
    errors="coerce",
)

print("Start date:", invoice_dates.min())
print("End date:", invoice_dates.max())
print("Unparseable dates:", int(invoice_dates.isna().sum()))


Start date: 2010-12-01 08:26:00
End date: 2011-12-09 12:50:00
Unparseable dates: 0


## 13. Data Quality Summary

In [29]:
data_quality_summary = pd.DataFrame(
    {
        "metric": [
            "rows",
            "columns",
            "unique_invoices",
            "unique_customers",
            "unique_stock_codes",
            "countries",
            "missing_customer_id",
            "missing_description",
            "duplicate_rows",
            "quantity_negative",
            "quantity_zero",
            "unit_price_negative",
            "unit_price_zero",
            "cancelled_rows",
            "cancelled_invoices",
            "start_date",
            "end_date",
            "unparseable_invoice_dates",
        ],
        "value": [
            len(df),
            df.shape[1],
            unique_invoice_count,
            unique_customer_count,
            unique_stock_codes,
            country_count,
            int(df["CustomerID"].isna().sum()),
            int(df["Description"].isna().sum()),
            duplicate_count,
            int((df["Quantity"] < 0).sum()),
            int((df["Quantity"] == 0).sum()),
            int((df["UnitPrice"] < 0).sum()),
            int((df["UnitPrice"] == 0).sum()),
            cancelled_row_count,
            cancelled_invoice_count,
            invoice_dates.min(),
            invoice_dates.max(),
            int(invoice_dates.isna().sum()),
        ],
    }
)

data_quality_summary


,metric,value
0,rows,541909
1,columns,8
2,unique_invoices,25900
3,unique_customers,4372
4,unique_stock_codes,4070
5,countries,38
6,missing_customer_id,135080
7,missing_description,1454
8,duplicate_rows,5268
9,quantity_negative,10624


## 14. Findings before cleaning

Tự ghi câu trả lời:

1. Cột nào có missing values?
2. Missing `CustomerID` ảnh hưởng sales analysis và customer analysis khác nhau thế nào?
3. Duplicate rows có tồn tại không? Có chắc nên xóa tất cả không?
4. Quan hệ giữa cancelled invoices và negative Quantity là gì?
5. Có `UnitPrice <= 0` không? Các row đó có đặc điểm gì?
6. `InvoiceDate` hiện có dtype gì? Có parse lỗi không?
7. Một invoice có nhiều rows không?
8. Có StockCode nào có nhiều Description không?
9. UK chiếm tỷ trọng dữ liệu lớn đến mức nào?
10. Vấn đề nào là structural error và vấn đề nào là data-quality/business issue?

> Chỉ sau khi trả lời được các câu trên mới thiết kế cleaning strategy.
